In [1]:
from torch.utils.data import Dataset, DataLoader
from common import load_titanic

X_train, X_val, y_train, y_val = load_titanic()
class TitanicDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = TitanicDataset(X_train, y_train)
val_ds = TitanicDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)


In [2]:
import numpy as np
import torch.nn as nn
import torch
from common import Net, accuracy
import copy


X_train, X_val, y_train, y_val = load_titanic()


torch.manual_seed(42)
model = Net(6)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
best_val, best_state, wait, patience = float("inf"), None, 0, 10

for epoch in range(50):
    model.train()
    train_losses = []
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses, correct, total = [], 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            val_losses.append(criterion(logits, y_batch).item())
            correct += ((logits > 0).float() == y_batch).sum().item()
            total += len(y_batch)

    val_loss = sum(val_losses) / len(val_losses)

    if val_loss < best_val:
        best_val = val_loss
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"early stop на епосі {epoch}, найкращий val={best_val:.4f}")
            model.load_state_dict(best_state)
            print("фінальна accuracy:", accuracy(model, X_val, y_val))
            break


    if epoch % 10 == 0:
        print(f"epoch {epoch:2d}  train={np.mean(train_losses):.4f}  "
              f"val={np.mean(val_losses):.4f}  acc={correct/total:.3f}")



from common import load_titanic, run_seeds

X_train, X_val, y_train, y_val = load_titanic()

# завдання 2 — вплив batch_size
for bs in [8, 32, 128, len(X_train)]:
    mean, std = run_seeds(X_train, y_train, X_val, y_val, batch_size=bs)
    print(f"batch_size={bs:4d}  {mean:.3f} ± {std:.3f}")

# завдання 3 — без перемішування
print("shuffle=False:", run_seeds(X_train, y_train, X_val, y_val, shuffle=False))

# завдання 4 — gradient accumulation
print("bs=32:          ", run_seeds(X_train, y_train, X_val, y_val, batch_size=32))
print("bs=8, accum=4:  ", run_seeds(X_train, y_train, X_val, y_val, batch_size=8, accum=4))



epoch  0  train=0.6759  val=0.6646  acc=0.615
epoch 10  train=0.4254  val=0.4560  acc=0.827
epoch 20  train=0.4021  val=0.4413  acc=0.804
epoch 30  train=0.4165  val=0.4424  acc=0.788
early stop на епосі 32, найкращий val=0.4399
фінальна accuracy: 0.8156424760818481
batch_size=   8  0.828 ± 0.013
batch_size=  32  0.822 ± 0.014
batch_size= 128  0.818 ± 0.013
batch_size= 712  0.811 ± 0.004
shuffle=False: (0.8145251393318176, 0.006515044840765326)
bs=32:           (0.8223463773727417, 0.013865568281995425)
bs=8, accum=4:   (0.813407814502716, 0.01395531392305567)
